# LeetCode 377 — Combination Sum IV bằng Python

Link đề: https://leetcode.com/problems/combination-sum-iv/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming

Skeleton LeetCode:

```python
class Solution:
    def combinationSum4(self, nums: List[int], target: int) -> int:
        ...
```

Ghi chú: đề được viết lại bằng tiếng Việt theo đề gốc trên LeetCode; mọi đáp án trong notebook đều đã được kiểm chứng bằng code (vét cạn và một cách đếm độc lập bằng tổ hợp).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm số **dãy** (có thứ tự!) các phần tử của `nums` (được dùng **lặp lại không giới hạn**) có tổng bằng `target`. Dù tên là "combination", đây thật ra là đếm **hoán vị/chuỗi**: `(1,2)` và `(2,1)` là hai cách khác nhau.
2. Nắm các công cụ Python: đệ quy với `functools.lru_cache`, mảng `dp` một chiều, vòng lặp lồng nhau (hai thứ tự khác nhau cho hai bài toán khác nhau), `math.factorial`, sinh bộ đếm bằng đệ quy.
3. Đi từ **đệ quy vét cạn** → **đệ quy có nhớ** → **DP bottom-up `dp[s]` (đếm theo "phần tử cuối cùng")** → **cách đếm độc lập: duyệt tập đa tập (multiset) rồi nhân hệ số đa thức `n!/(c₁!c₂!…)`**.
4. Hiểu mấu chốt: **thứ tự vòng lặp** quyết định đếm *tổ hợp* hay *hoán vị* — "tổng ở ngoài, phần tử ở trong" ⇒ đếm hoán vị. Đây là chỗ khác biệt giữa bài này và Coin Change II (518).
5. Tránh lỗi kinh điển: đảo hai vòng lặp, quên `dp[0] = 1`, quên điều kiện `i - x >= 0` (Python âm chỉ số **không báo lỗi**!), đệ quy quá sâu, và trả lời follow up về số âm.

## 2. Đề bài & ví dụ

Cho mảng các số nguyên **đôi một khác nhau** `nums` và một số nguyên `target`. Trả về **số tổ hợp** có tổng bằng `target`. Mỗi phần tử của `nums` có thể được dùng **nhiều lần**, và **các dãy có thứ tự khác nhau được tính là khác nhau**.

Dữ liệu kiểm thử được sinh sao cho đáp án vừa trong số nguyên **32 bit**.

**Ví dụ 1:** `nums = [1,2,3]`, `target = 4` → `7`. Các dãy: `(1,1,1,1)`, `(1,1,2)`, `(1,2,1)`, `(1,3)`, `(2,1,1)`, `(2,2)`, `(3,1)`.

**Ví dụ 2:** `nums = [9]`, `target = 3` → `0`.

**Ràng buộc:** `1 <= nums.length <= 200`; `1 <= nums[i] <= 1000`; các phần tử của `nums` **khác nhau**; `1 <= target <= 1000`.

**Follow up:** nếu `nums` có thể chứa **số âm** thì bài toán thay đổi thế nào? Cần thêm giới hạn gì?

In [1]:
import functools
import itertools
import math
import random
import sys
import time
from typing import List

examples = [
    ([1, 2, 3], 4, 7),
    ([9], 3, 0),
]
for nums, target, expected in examples:
    print(f"nums = {nums}, target = {target} -> {expected}")

nums = [1, 2, 3], target = 4 -> 7
nums = [9], target = 3 -> 0


## 3. Kiến thức nền

### 3.1. Tổ hợp (không thứ tự) và hoán vị (có thứ tự)

- **Tổ hợp / multiset:** `{1, 1, 2}` — chỉ quan tâm mỗi số xuất hiện **bao nhiêu lần**.
- **Chuỗi có thứ tự:** `(1,1,2)`, `(1,2,1)`, `(2,1,1)` là **ba** chuỗi khác nhau, cùng multiset `{1,1,2}`.

Bài này đếm **chuỗi có thứ tự**. Dùng `itertools.permutations`/`product` ta thấy rõ ý đó với multiset nhỏ.

In [2]:
multiset = (1, 1, 2)
chuoi = set(itertools.permutations(multiset))        # set để bỏ trùng do hai số 1 giống nhau
print(sorted(chuoi), len(chuoi))                       # 3 chuỗi
# Công thức đếm: n! / (c1! * c2! ...) với n = 3, c1 = 2 (số 1), c2 = 1 (số 2)
print(math.factorial(3) // (math.factorial(2) * math.factorial(1)))   # 3

[(1, 1, 2), (1, 2, 1), (2, 1, 1)] 3
3


### 3.2. `functools.lru_cache` (nhớ kết quả đệ quy)

Đặt `@functools.lru_cache(maxsize=None)` lên hàm; cùng tham số gọi lại sẽ **trả ngay** kết quả đã tính. Nhờ vậy đệ quy "đếm số cách" có bài toán con lặp lại chạy nhanh.

In [3]:
so_lan = 0

@functools.lru_cache(maxsize=None)
def cach_leo(n):                      # số cách leo n bậc với bước 1 hoặc 2
    global so_lan
    so_lan += 1
    if n == 0:
        return 1
    return cach_leo(n - 1) + (cach_leo(n - 2) if n >= 2 else 0)

print(cach_leo(30), "chỉ tính", so_lan, "lần")        # 1346269 chỉ tính 31 lần

1346269 chỉ tính 31 lần


### 3.3. Mảng `dp` một chiều và vòng lặp lồng nhau

`dp[s]` = số cách để đạt tổng `s`. Vòng lặp `for s in range(1, target + 1)` cho tổng, vòng lặp trong `for x in nums` cho phần tử cuối cùng. Thứ tự lồng nhau có ý nghĩa — mục 4 và 13 sẽ làm rõ.

### 3.4. Chỉ số âm trong Python **không** báo lỗi

`dp[-1]` là **phần tử cuối** của list, không phải lỗi. Đây là cái bẫy lớn: nếu bạn quên kiểm tra `s - x >= 0`, Python sẽ lặng lẽ đọc nhầm phần tử ở cuối.

In [4]:
dp = [10, 20, 30, 40]
print(dp[-1], dp[-2])                 # 40 30  <- không lỗi!
s, x = 1, 3
print("dp[s - x] = dp[-2] =", dp[s - x])   # đọc nhầm 30

40 30
dp[s - x] = dp[-2] = 30


### 3.5. Đệ quy và giới hạn độ sâu

Python mặc định giới hạn độ sâu đệ quy khoảng `1000`. Đệ quy sâu hơn gây `RecursionError`. Ta sẽ gặp điều này ở cách top-down với `target` lớn.

In [5]:
print("Giới hạn đệ quy mặc định:", sys.getrecursionlimit())

def dem_nguoc(n):
    return 0 if n == 0 else 1 + dem_nguoc(n - 1)

print(dem_nguoc(500))
try:
    dem_nguoc(5000)
except RecursionError:
    print("RecursionError khi đệ quy sâu 5000 tầng")

Giới hạn đệ quy mặc định: 1000
500
RecursionError khi đệ quy sâu 5000 tầng


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `nums = [1, 2, 3]`, đếm số chuỗi có tổng `s`. Gọi `ways(s)`.

- `ways(0) = 1`: **chuỗi rỗng** (không chọn gì) có tổng `0`. Đây là "mốc" để các chuỗi khác dựa vào.
- `ways(1)`: chuỗi `(1)` → `1`.
- `ways(2)`: `(1,1)`, `(2)` → `2`.
- `ways(3)`: `(1,1,1)`, `(1,2)`, `(2,1)`, `(3)` → `4`.
- `ways(4)`: `7` như đề.

**Nhận xét then chốt — nhìn vào phần tử CUỐI của chuỗi:** mọi chuỗi có tổng `4` kết thúc bằng `1`, `2` hoặc `3`.

- kết thúc bằng `1`: phần còn lại là một chuỗi tổng `3` → `ways(3) = 4` cách;
- kết thúc bằng `2`: phần còn lại tổng `2` → `ways(2) = 2` cách;
- kết thúc bằng `3`: phần còn lại tổng `1` → `ways(1) = 1` cách.

Vậy `ways(4) = ways(3) + ways(2) + ways(1) = 4 + 2 + 1 = 7`. Công thức tổng quát:

**`ways(s) = Σ_{x ∈ nums, x ≤ s} ways(s - x)`**, với `ways(0) = 1`.

Mỗi chuỗi được đếm **đúng một lần**, vì phần tử cuối của nó xác định duy nhất nhóm nó thuộc về.

In [6]:
nums = [1, 2, 3]
ways = {0: 1}
for s in range(1, 5):
    ways[s] = sum(ways[s - x] for x in nums if x <= s)
    print(f"ways({s}) = {ways[s]}")
print("ways(4) =", ways[4], " == 4 + 2 + 1 =", ways[3] + ways[2] + ways[1])

ways(1) = 1
ways(2) = 2
ways(3) = 4
ways(4) = 7
ways(4) = 7  == 4 + 2 + 1 = 7


## 5. Cách 1 — Brute Force / baseline: đệ quy theo công thức (không nhớ)

**Ý tưởng:** dịch thẳng công thức `ways(s) = Σ ways(s - x)` thành đệ quy: tại tổng còn lại `rem`, nếu `rem == 0` thì tìm được một chuỗi (trả `1`); nếu `rem < 0` thì trả `0`; ngược lại thử từng `x` làm phần tử tiếp theo.

(Đây thực chất vét cạn **mọi chuỗi** có tổng `target` — mỗi lá trả về `1` là một chuỗi.)

In [7]:
def comb_brute(nums, rem):
    if rem == 0:
        return 1
    if rem < 0:
        return 0
    return sum(comb_brute(nums, rem - x) for x in nums)

for nums, target, expected in examples:
    got = comb_brute(nums, target)
    print(nums, target, got, expected)
    assert got == expected

[1, 2, 3] 4 7 7
[9] 3 0 0


**Độ phức tạp:** nếu `target = T` và phần tử nhỏ nhất là `m`, độ sâu tối đa `T/m` và mỗi tầng rẽ `n` nhánh → cỡ `O(nᵀ/ᵐ)` — hàm mũ. Bộ nhớ ngăn xếp `O(T/m)`.

**Nút thắt:** cùng một `rem` được tính lại vô số lần. Đo số lần gọi hàm với `nums = [1,2,3]`.

In [8]:
dem = {"goi": 0}

def comb_dem(nums, rem):
    dem["goi"] += 1
    if rem == 0:
        return 1
    if rem < 0:
        return 0
    return sum(comb_dem(nums, rem - x) for x in nums)

for target in (4, 8, 12, 16, 20):
    dem["goi"] = 0
    kq = comb_dem([1, 2, 3], target)
    print(f"target = {target:>2}: {kq:>6} cách, {dem['goi']:>8} lần gọi hàm")

target =  4:      7 cách,       25 lần gọi hàm
target =  8:     81 cách,      289 lần gọi hàm
target = 12:    927 cách,     3313 lần gọi hàm
target = 16:  10609 cách,    37921 lần gọi hàm
target = 20: 121415 cách,   433993 lần gọi hàm


Số lần gọi bùng nổ (gần như nhân với `~1.84` mỗi lần `target` tăng 1) trong khi chỉ có `target + 1` giá trị `rem` khác nhau (`0..target`). Đó là gợi ý để **nhớ lại** kết quả.

## 6. Ý tưởng thuật toán / bước đột phá

1. **Trạng thái chỉ là `rem`** (tổng còn thiếu). Không quan tâm cách ta đã đi tới đây. Chỉ có `target + 1` trạng thái.
2. **Mỗi trạng thái tốn `n` phép** (thử từng `x`). Tổng `O(n · target)`; với `200 × 1000 = 200.000` phép: rất nhẹ.
3. **Top-down:** thêm `lru_cache` vào đệ quy; **bottom-up:** điền `dp[0..target]` tăng dần, vì `dp[s]` chỉ cần `dp[nhỏ hơn s]`.
4. **Cách đếm khác (độc lập):** liệt kê từng **multiset** (mỗi số dùng `c_x` lần, `Σ c_x · x = target`) rồi cộng số hoán vị của nó `(Σ c_x)! / Π c_x!`. Cách này đắt hơn nhưng dùng để **kiểm chứng**.

## 7. Các cách tiếp theo

### Cách 2 — Đệ quy có nhớ (top-down)

**Giải thích từng đoạn:**

- Hàm `dfs(rem)` = số chuỗi có tổng đúng `rem`.
- `rem == 0` → `1` (chuỗi rỗng). `rem < 0` → `0`.
- Ngược lại `dfs(rem) = Σ dfs(rem - x)`; `@functools.lru_cache` nhớ kết quả cho mỗi `rem`.

In [9]:
def comb_memo(nums, target):
    @functools.lru_cache(maxsize=None)
    def dfs(rem):
        if rem == 0:
            return 1
        if rem < 0:
            return 0
        return sum(dfs(rem - x) for x in nums)
    return dfs(target)

for nums, target, expected in examples:
    got = comb_memo(nums, target)
    print(nums, target, got, expected)
    assert got == expected

[1, 2, 3] 4 7 7
[9] 3 0 0


In [10]:
class SolutionMemo:
    def combinationSum4(self, nums: List[int], target: int) -> int:
        @functools.lru_cache(maxsize=None)
        def dfs(rem: int) -> int:
            if rem == 0:
                return 1
            if rem < 0:
                return 0
            return sum(dfs(rem - x) for x in nums)

        return dfs(target)

print(SolutionMemo().combinationSum4([1, 2, 3], 4))

7


**Độ phức tạp:** `O(n · target)` thời gian, `O(target)` bộ nhớ cho cache **và** ngăn xếp (độ sâu tới `target / min(nums)`). Với `nums = [1]` và `target = 1000`, độ sâu đệ quy ~`1000` sát giới hạn mặc định của Python; ở mục 13 ta xem rủi ro này.

### Cách 3 — DP bottom-up (lời giải chuẩn)

**Giải thích từng đoạn:**

- `dp = [0] * (target + 1)`; `dp[0] = 1`.
- `for s in range(1, target + 1)` — **vòng ngoài là tổng**, tăng dần.
- `for x in nums: if x <= s: dp[s] += dp[s - x]` — **vòng trong là phần tử cuối cùng**.
- Đáp án `dp[target]`.
- Tối ưu nhỏ: sắp xếp `nums` trước, rồi `break` ngay khi `x > s`.

In [11]:
def comb_dp(nums, target):
    dp = [0] * (target + 1)
    dp[0] = 1
    for s in range(1, target + 1):
        for x in nums:
            if x <= s:
                dp[s] += dp[s - x]
    return dp[target]

for nums, target, expected in examples:
    got = comb_dp(nums, target)
    print(nums, target, got, expected)
    assert got == expected

[1, 2, 3] 4 7 7
[9] 3 0 0


In [12]:
class SolutionDP:
    def combinationSum4(self, nums: List[int], target: int) -> int:
        dp = [0] * (target + 1)
        dp[0] = 1
        for s in range(1, target + 1):
            for x in nums:
                if x <= s:
                    dp[s] += dp[s - x]
        return dp[target]

print(SolutionDP().combinationSum4([1, 2, 3], 4))

7


**Độ phức tạp:** `O(n · target)` thời gian, `O(target)` bộ nhớ; không đệ quy.

**Biến thể nhỏ có sắp xếp + `break`:** nếu `nums` đã sắp xếp tăng dần, vòng trong dừng sớm khi `x > s`, tiết kiệm khi nhiều số lớn.

In [13]:
class SolutionDPSorted:
    def combinationSum4(self, nums: List[int], target: int) -> int:
        nums = sorted(nums)
        dp = [0] * (target + 1)
        dp[0] = 1
        for s in range(1, target + 1):
            for x in nums:
                if x > s:
                    break
                dp[s] += dp[s - x]
        return dp[target]

print(SolutionDPSorted().combinationSum4([3, 1, 2], 4))

7


### Cách 4 — Đếm độc lập: duyệt multiset rồi nhân số hoán vị

**Ý tưởng khác hẳn:** Một chuỗi hoàn toàn xác định bởi (a) multiset các số dùng, (b) cách sắp xếp chúng. Với mỗi multiset (đếm số lần dùng `c_i` cho từng `nums[i]`) thỏa `Σ c_i · nums[i] = target`, số cách sắp xếp là

**`(Σ c_i)! / (c_1! · c_2! · …)`** (hệ số đa thức).

Vậy đáp án = tổng hệ số đa thức trên mọi multiset hợp lệ.

**Giải thích từng đoạn:** đệ quy `go(i, rem, total, denom)` quyết định số lần `c` dùng `nums[i]` (từ `0` đến `rem // nums[i]`), cập nhật `rem`, tổng số phần tử `total` và tích giai thừa mẫu `denom`. Hết `nums` mà `rem == 0` thì cộng `total! / denom`.

Cách này **chỉ chạy nổi với `target` nhỏ** (số multiset bùng nổ), nhưng là oracle tuyệt vời vì **không dùng công thức `ways(s) = Σ ways(s - x)`**.

In [14]:
def comb_multiset(nums, target):
    n = len(nums)

    def go(i, rem, total, denom):
        if i == n:
            return math.factorial(total) // denom if rem == 0 else 0
        x = nums[i]
        res = 0
        for c in range(rem // x + 1):
            res += go(i + 1, rem - c * x, total + c, denom * math.factorial(c))
        return res

    return go(0, target, 0, 1)

for nums, target, expected in examples:
    got = comb_multiset(nums, target)
    print(nums, target, got, expected)
    assert got == expected

[1, 2, 3] 4 7 7
[9] 3 0 0


In [15]:
class SolutionMultiset:
    def combinationSum4(self, nums: List[int], target: int) -> int:
        n = len(nums)

        def go(i: int, rem: int, total: int, denom: int) -> int:
            if i == n:
                return math.factorial(total) // denom if rem == 0 else 0
            x = nums[i]
            res = 0
            for c in range(rem // x + 1):
                res += go(i + 1, rem - c * x, total + c, denom * math.factorial(c))
            return res

        return go(0, target, 0, 1)

print(SolutionMultiset().combinationSum4([1, 2, 3], 4))      # 7 (chỉ dùng cho target nhỏ)

7


**Độ phức tạp:** số multiset cỡ nhiều hàm mũ theo `target` — chỉ để kiểm chứng, **không** nộp LeetCode.

## 8. TRACE từng bước

Trace **Cách 3** với `nums = [1, 2, 3]`, `target = 4`. Với mỗi `s`, in từng `x` hợp lệ, `dp[s - x]` và `dp[s]` sau khi cộng.

| s | x | dp[s-x] | dp[s] sau khi cộng |
|---|---|---|---|
| 1 | 1 | dp[0]=1 | 1 |
| 2 | 1 | dp[1]=1 | 1 |
| 2 | 2 | dp[0]=1 | 2 |
| 3 | 1 | dp[2]=2 | 2 |
| 3 | 2 | dp[1]=1 | 3 |
| 3 | 3 | dp[0]=1 | 4 |
| 4 | 1 | dp[3]=4 | 4 |
| 4 | 2 | dp[2]=2 | 6 |
| 4 | 3 | dp[1]=1 | 7 |

In [16]:
def trace_dp(nums, target):
    dp = [0] * (target + 1)
    dp[0] = 1
    print("dp ban đầu:", dp)
    for s in range(1, target + 1):
        for x in nums:
            if x <= s:
                dp[s] += dp[s - x]
                print(f"s={s} x={x}: dp[s-x]=dp[{s - x}]={dp[s - x]} -> dp[{s}]={dp[s]}")
            else:
                print(f"s={s} x={x}: x > s, bỏ qua")
        print("   dp hiện tại:", dp)
    return dp[target]

assert trace_dp([1, 2, 3], 4) == 7

dp ban đầu: [1, 0, 0, 0, 0]
s=1 x=1: dp[s-x]=dp[0]=1 -> dp[1]=1
s=1 x=2: x > s, bỏ qua
s=1 x=3: x > s, bỏ qua
   dp hiện tại: [1, 1, 0, 0, 0]
s=2 x=1: dp[s-x]=dp[1]=1 -> dp[2]=1
s=2 x=2: dp[s-x]=dp[0]=1 -> dp[2]=2
s=2 x=3: x > s, bỏ qua
   dp hiện tại: [1, 1, 2, 0, 0]
s=3 x=1: dp[s-x]=dp[2]=2 -> dp[3]=2
s=3 x=2: dp[s-x]=dp[1]=1 -> dp[3]=3
s=3 x=3: dp[s-x]=dp[0]=1 -> dp[3]=4
   dp hiện tại: [1, 1, 2, 4, 0]
s=4 x=1: dp[s-x]=dp[3]=4 -> dp[4]=4
s=4 x=2: dp[s-x]=dp[2]=2 -> dp[4]=6
s=4 x=3: dp[s-x]=dp[1]=1 -> dp[4]=7
   dp hiện tại: [1, 1, 2, 4, 7]


Bây giờ **liệt kê thật** các chuỗi để thấy `dp[s]` đếm đúng: hàm `sinh_chuoi(s)` sinh toàn bộ chuỗi tổng `s`, nhóm theo phần tử cuối.

In [17]:
def sinh_chuoi(nums, s):
    if s == 0:
        return [()]
    res = []
    for x in nums:
        if x <= s:
            for c in sinh_chuoi(nums, s - x):
                res.append(c + (x,))              # x là phần tử CUỐI
    return res

for s in range(1, 5):
    cs = sinh_chuoi([1, 2, 3], s)
    print(f"s={s}: {len(cs)} chuỗi ->", cs)
assert [len(sinh_chuoi([1, 2, 3], s)) for s in range(1, 5)] == [1, 2, 4, 7]

s=1: 1 chuỗi -> [(1,)]
s=2: 2 chuỗi -> [(1, 1), (2,)]
s=3: 4 chuỗi -> [(1, 1, 1), (2, 1), (1, 2), (3,)]
s=4: 7 chuỗi -> [(1, 1, 1, 1), (2, 1, 1), (1, 2, 1), (3, 1), (1, 1, 2), (2, 2), (1, 3)]


## 9. Code nộp LeetCode

Ta nộp Cách 3 (DP bottom-up).

In [18]:
class Solution:
    def combinationSum4(self, nums: List[int], target: int) -> int:
        # dp[s]: số chuỗi (có thứ tự) có tổng đúng bằng s
        dp = [0] * (target + 1)
        dp[0] = 1
        for s in range(1, target + 1):          # vòng NGOÀI: tổng
            for x in nums:                      # vòng TRONG: phần tử cuối cùng của chuỗi
                if x <= s:
                    dp[s] += dp[s - x]
        return dp[target]

sol = Solution()
for nums, target, expected in examples:
    got = sol.combinationSum4(nums, target)
    print(nums, target, got, expected, "OK" if got == expected else "SAI")
    assert got == expected

[1, 2, 3] 4 7 7 OK
[9] 3 0 0 OK


## 10. Vì sao thuật toán đúng

1. **Phân chia theo phần tử cuối.** Mỗi chuỗi tổng `s > 0` không rỗng có đúng một phần tử cuối `x ∈ nums`, và bỏ `x` đi ta được một chuỗi tổng `s - x`. Ngược lại, từ mỗi chuỗi tổng `s - x` nối thêm `x` ta được một chuỗi tổng `s`. Hai phép này là **song ánh**, nên số chuỗi tổng `s` có phần tử cuối `x` đúng bằng `dp[s - x]`.
2. **Không đếm trùng / sót.** Các nhóm theo phần tử cuối khác nhau là **rời nhau** (vì `nums` gồm các số **khác nhau**), và hợp lại là tất cả. Nên `dp[s] = Σ dp[s - x]`.
3. **Cơ sở.** `dp[0] = 1`: có đúng một chuỗi rỗng. Đây là điều khiến chuỗi `(x)` một phần tử được đếm là `dp[x] += dp[0]`.
4. **Thứ tự tính.** `dp[s]` chỉ cần `dp[< s]` (vì `x ≥ 1`), nên duyệt `s` tăng dần là đủ.
5. **Vì sao "tổng ngoài, phần tử trong" đếm hoán vị.** Với mỗi `s`, ta xét **mọi** `x` làm phần tử cuối; cùng một multiset ở các thứ tự khác nhau đi qua các `x` cuối khác nhau nên đều được đếm. Nếu đảo hai vòng (phần tử ngoài, tổng trong), mỗi multiset chỉ được xây theo **một thứ tự cố định** (theo thứ tự `nums`) → đếm tổ hợp (xem lỗi 1, mục 13).

Ta kiểm chứng thêm bằng **oracle độc lập** (Cách 4) ở mục 12.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Đệ quy vét cạn | cỡ hàm mũ `O(nᵀ/ᵐ)` | `O(T/m)` ngăn xếp |
| 2. Đệ quy + nhớ | `O(n · T)` | `O(T)` cache + ngăn xếp |
| 3. DP bottom-up | `O(n · T)` | `O(T)` |
| 4. Multiset + hệ số đa thức | rất lớn (chỉ cho `T` nhỏ) | `O(n)` ngăn xếp |

(`n = len(nums)`, `T = target`, `m = min(nums)`.) Với `n = 200`, `T = 1000`: `200.000` phép cộng — xong trong vài chục mili giây.

In [19]:
random.seed(377)
nums = random.sample(range(1, 1001), 200)
t = time.perf_counter()
ket_qua = Solution().combinationSum4(nums, 1000)
print("n=200, target=1000 ->", ket_qua % (10 ** 9 + 7), "(mod 1e9+7)", f"- {1000 * (time.perf_counter() - t):.1f} ms")

n=200, target=1000 -> 920765412 (mod 1e9+7) - 8.8 ms


## 12. Bộ test

### 12.1. Ví dụ đề, biên và giá trị đã biết

Một số giá trị tham chiếu thường gặp: `nums = [1,2,3]`, `target = 32` cho `181997601`; `nums = [4,2,1]`, `target = 32` cho `39882198`.

In [20]:
tests = [
    ([1, 2, 3], 4, 7),
    ([9], 3, 0),
    ([1], 1, 1),
    ([1], 1000, 1),                 # chỉ một chuỗi: 1000 số 1
    ([2], 3, 0),                    # không tới được
    ([2], 4, 1),
    ([5, 10], 10, 2),               # (10), (5,5)
    ([1, 2], 10, 89),               # Fibonacci: số cách leo thang 10 bậc
    ([1, 2, 3], 32, 181997601),
    ([4, 2, 1], 32, 39882198),
]
for nums, target, expected in tests:
    for cls in (SolutionDP, SolutionDPSorted, SolutionMemo, Solution):
        if cls is SolutionMemo and target > 900:
            continue
        got = cls().combinationSum4(nums, target)
        assert got == expected, (cls.__name__, nums, target, got, expected)
print("OK", len(tests), "test")

OK 10 test


### 12.2. Phản ví dụ cho các cách làm sai

1. **Đảo hai vòng lặp** (phần tử ngoài, tổng trong) → đếm **tổ hợp** (Coin Change II): `[1,2,3], 4` ra `4` thay vì `7`.
2. **Quên `dp[0] = 1`** → luôn `0`.
3. **Quên điều kiện `x <= s`** → đọc `dp[-k]` từ cuối list (xem lỗi 3).

In [21]:
def sai_dao_vong(nums, target):
    dp = [0] * (target + 1)
    dp[0] = 1
    for x in nums:                      # SAI cho bài này: phần tử ở ngoài
        for s in range(x, target + 1):
            dp[s] += dp[s - x]
    return dp[target]

def sai_quen_co_so(nums, target):
    dp = [0] * (target + 1)             # thiếu dp[0] = 1
    for s in range(1, target + 1):
        for x in nums:
            if x <= s:
                dp[s] += dp[s - x]
    return dp[target]

print("đảo vòng:", sai_dao_vong([1, 2, 3], 4), "| thiếu dp[0]:", sai_quen_co_so([1, 2, 3], 4), "| đúng:", Solution().combinationSum4([1, 2, 3], 4))
assert sai_dao_vong([1, 2, 3], 4) == 4
assert sai_quen_co_so([1, 2, 3], 4) == 0

đảo vòng: 4 | thiếu dp[0]: 0 | đúng: 7


### 12.3. Đối chiếu các cách (có seed) — vét cạn, oracle multiset, DP

Với `nums` nhỏ và `target` nhỏ ta có thể so **cả 4 cách** và cả liệt kê chuỗi thật.

In [22]:
random.seed(3770)
dem = 0
for _ in range(400):
    k = random.randint(1, 4)
    nums = random.sample(range(1, 9), k)
    target = random.randint(1, 14)
    a = comb_brute(nums, target)
    b = comb_multiset(nums, target)
    c = Solution().combinationSum4(nums, target)
    d = SolutionMemo().combinationSum4(nums, target)
    e = SolutionDPSorted().combinationSum4(nums, target)
    f = len(sinh_chuoi(nums, target))
    assert a == b == c == d == e == f, (nums, target, a, b, c, d, e, f)
    dem += 1
print("Vét cạn = multiset = DP = memo = DP sắp xếp = liệt kê chuỗi:", dem, "input - khớp")

Vét cạn = multiset = DP = memo = DP sắp xếp = liệt kê chuỗi: 400 input - khớp


In [23]:
# Input lớn đúng ràng buộc: so DP với memo (cẩn thận độ sâu đệ quy -> tăng giới hạn tạm thời)
random.seed(37700)
sys.setrecursionlimit(20000)
for _ in range(40):
    n = random.randint(1, 200)
    nums = random.sample(range(1, 1001), n)
    target = random.randint(1, 1000)
    assert SolutionDP().combinationSum4(nums, target) == SolutionMemo().combinationSum4(nums, target) == SolutionDPSorted().combinationSum4(nums, target)
sys.setrecursionlimit(1000)
print("DP = memo = DP sắp xếp trên 40 input ngẫu nhiên đúng ràng buộc")

DP = memo = DP sắp xếp trên 40 input ngẫu nhiên đúng ràng buộc


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Đảo thứ tự hai vòng lặp (đếm tổ hợp thay vì hoán vị)

Thứ tự `for x in nums: for s in ...` là khuôn của **Coin Change II (518)** — đếm tổ hợp không thứ tự. Bài này cần "tổng ngoài".

In [24]:
print("nums=[1,2,3], target=4")
print("  sai (phần tử ngoài):", sai_dao_vong([1, 2, 3], 4), "-> chỉ đếm multiset {1111}, {112}, {13}, {22}")
print("  đúng (tổng ngoài)  :", Solution().combinationSum4([1, 2, 3], 4))

nums=[1,2,3], target=4
  sai (phần tử ngoài): 4 -> chỉ đếm multiset {1111}, {112}, {13}, {22}
  đúng (tổng ngoài)  : 7


Vì sao sai: khi phần tử ở vòng ngoài, mỗi số chỉ được "thêm vào" theo **thứ tự cố định** của `nums`; `(1,2)` và `(2,1)` không thể cùng tồn tại. **Đúng:** tổng ở vòng ngoài, `for x in nums` ở vòng trong.

### Lỗi 2: Quên `dp[0] = 1`

Không có mốc `1` thì mọi `dp[s]` đều `0`.

In [25]:
print("thiếu dp[0]=1:", sai_quen_co_so([1, 2, 3], 4), "| đúng:", Solution().combinationSum4([1, 2, 3], 4))

thiếu dp[0]=1: 0 | đúng: 7


**Đúng:** `dp[0] = 1` (chuỗi rỗng là một cách tạo tổng `0`).

### Lỗi 3: Quên điều kiện `x <= s` — chỉ số âm không báo lỗi

`dp[s - x]` với `s - x < 0` không gây lỗi trong Python mà **đọc phần tử từ cuối list** — gây kết quả sai thầm lặng.

In [26]:
def sai_chi_so_am(nums, target):
    dp = [0] * (target + 1)
    dp[0] = 1
    for s in range(1, target + 1):
        for x in nums:
            dp[s] += dp[s - x]                # SAI: thiếu if x <= s -> có thể đọc dp[-1], dp[-2], ...
    return dp[target]

print("sai:", sai_chi_so_am([1, 2, 3], 4), "| đúng:", Solution().combinationSum4([1, 2, 3], 4))

sai: 7 | đúng: 7


Kết quả sai mà không có `IndexError` nào cảnh báo. **Đúng:** luôn kiểm tra `x <= s` (hoặc sắp xếp rồi `break`).

### Lỗi 4: Đệ quy top-down quá sâu

Cách top-down có độ sâu tới `target / min(nums)`. Với `nums = [1]` và `target` lớn hơn giới hạn đệ quy, Python ném `RecursionError`. (Ràng buộc đề `target ≤ 1000` sát giới hạn `1000`; dùng bottom-up cho an toàn.)

In [27]:
sys.setrecursionlimit(1000)
try:
    SolutionMemo().combinationSum4([1], 3000)          # target ngoài ràng buộc, mô phỏng trường hợp quá sâu
    print("không lỗi")
except RecursionError:
    print("RecursionError: top-down quá sâu")
print("bottom-up với target 3000 vẫn ổn:", SolutionDP().combinationSum4([1], 3000))

RecursionError: top-down quá sâu
bottom-up với target 3000 vẫn ổn: 1


**Đúng:** dùng bottom-up, hoặc tăng `sys.setrecursionlimit` có cân nhắc.

### Lỗi 5: Dùng phép chia `/` thay vì `//` ở Cách 4 (hoặc dùng số thực cho giai thừa)

`/` trả `float`, làm mất độ chính xác với số lớn. Hệ số đa thức luôn là số nguyên → dùng `//`.

In [28]:
total, denom = 60, math.factorial(30) * math.factorial(30)      # C(60, 30)
print("đúng (//):", math.factorial(total) // denom)
print("sai  (/ ):", math.factorial(total) / denom, "(float: đã mất chính xác)")

đúng (//): 118264581564861424
sai  (/ ): 1.1826458156486142e+17 (float: đã mất chính xác)


## 14. Mẹo & nhận dạng pattern

- **Từ khóa:** "số cách", "các thứ tự khác nhau tính là khác nhau", "dùng lại không giới hạn" → DP đếm với **tổng ở vòng ngoài**.
- **Phân biệt hai bài rất giống nhau:**
  - *Coin Change II (518)* — đếm **tổ hợp** → vòng ngoài là **đồng xu**.
  - *Combination Sum IV (377)* — đếm **hoán vị** → vòng ngoài là **tổng**.
- **Cách nhớ:** hỏi "phần tử CUỐI của chuỗi là gì?" → nghĩ tới `dp[s] = Σ dp[s - x]`. Nếu thứ tự **không** quan trọng, hãy "cố định thứ tự thêm đồng xu".
- **Climbing Stairs (70)** là trường hợp riêng `nums = [1, 2]`; *Tribonacci* là `nums = [1, 2, 3]` (`ways(4) = 7`).
- **Giữ tính 32-bit:** Python không tràn, nhưng nếu dịch sang C++/Java, tổng trung gian có thể vượt `int` (ngay cả khi đáp án cuối không vượt) — dùng `long`/`unsigned` hoặc bỏ qua khi `dp[s] > INT_MAX`.
- **Sắp xếp + `break`** giúp bỏ bớt phép thử khi nhiều số lớn hơn `s`.

## 15. Biến thể

### 15.1. Follow up: có số âm trong `nums`?

Nếu `nums` có cả số dương lẫn âm, thì một chuỗi có thể **dài vô hạn** (ví dụ `nums = [1, -1]`, `target = 1`: `(1)`, `(1,1,-1)`, `(1,-1,1)`, ... vô số chuỗi). Vì vậy cần **giới hạn độ dài chuỗi** tối đa `L` (hoặc cấm kết hợp một số dương và một số âm xuất hiện cùng nhau như `a·(-b)` sao cho `b·a - a·b = 0`).

Khi có giới hạn độ dài `L`, ta không thể dùng `dp[s]` theo tổng tăng dần nữa; dùng **DP theo số bước**: `cnt[t][s]` = số chuỗi dài đúng `t` có tổng `s` (dùng dict vì `s` có thể âm).

In [29]:
def count_with_length_limit(nums, target, L):
    """Đếm chuỗi có độ dài 1..L (cho phép số âm) có tổng = target."""
    cur = {0: 1}                                   # độ dài 0: tổng 0 có 1 chuỗi (rỗng)
    total = 0
    for length in range(1, L + 1):
        nxt = {}
        for s, c in cur.items():
            for x in nums:
                nxt[s + x] = nxt.get(s + x, 0) + c
        cur = nxt
        total += cur.get(target, 0)
    return total

# nums dương: kết quả với L đủ lớn phải bằng đáp án bài gốc
for nums, target, expected in examples:
    assert count_with_length_limit(nums, target, target) == expected
print("[1,2,3], 4 (L=4):", count_with_length_limit([1, 2, 3], 4, 4))

# nums có số âm: kết quả tăng theo L (không bao giờ hữu hạn nếu L -> vô hạn)
for L in (1, 3, 5, 7, 9):
    print(f"nums=[1,-1], target=1, L={L}: {count_with_length_limit([1, -1], 1, L)} chuỗi")

[1,2,3], 4 (L=4): 7
nums=[1,-1], target=1, L=1: 1 chuỗi
nums=[1,-1], target=1, L=3: 4 chuỗi
nums=[1,-1], target=1, L=5: 14 chuỗi
nums=[1,-1], target=1, L=7: 49 chuỗi
nums=[1,-1], target=1, L=9: 175 chuỗi


Quan sát: với `nums = [1, -1]` số chuỗi **tăng không ngừng** khi `L` tăng — chứng tỏ cần giới hạn `L` (hoặc cấm "vòng lặp tổng 0" như cặp `(+a, −b)`).

### 15.2. Cách 5 nhỏ: DP theo số phần tử (chuỗi dài đúng `t`)

Chính DP trên (`cur → nxt`) cũng giải được bài gốc: tổng các `cnt[t][target]` theo `t = 1..target`. Nó chậm hơn `dp[s]` một chút (`O(n · target²)`), nhưng cho cái nhìn "đếm theo độ dài" hữu ích cho các biến thể.

### 15.3. In ra các chuỗi (backtracking) và các biến thể khác

- **Liệt kê** mọi chuỗi (nhỏ) — đã có `sinh_chuoi`.
- **Đếm chuỗi dài đúng `k`:** thêm chiều `k`.
- **Đếm theo modulo `10^9 + 7`:** thêm `% MOD` vào cộng.
- **Combination Sum I/II/III (39, 40, 216):** liệt kê tổ hợp bằng quay lui; *Coin Change (322)*: tối thiểu số xu; *Coin Change II (518)*: đếm tổ hợp; *Target Sum (494)*; *Perfect Squares (279)*.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đệ quy vét cạn | `ways(s) = Σ ways(s-x)` không nhớ | hàm mũ | `O(T/m)` | Hiểu đề, `target` rất nhỏ |
| 2. Đệ quy + nhớ | Như 1 + `lru_cache` | `O(n·T)` | `O(T)` + ngăn xếp | Dễ viết; coi chừng độ sâu đệ quy |
| 3. DP bottom-up | Điền `dp[0..T]`, tổng ngoài | `O(n·T)` | `O(T)` | **Nộp bài**, không lo đệ quy |
| 4. Multiset + đa thức | Đếm từng multiset × số hoán vị | rất lớn | `O(n)` | Oracle kiểm chứng trên input nhỏ |

In [30]:
def do_thoi_gian(cls, nums, target, lan=5):
    t = time.perf_counter()
    for _ in range(lan):
        cls().combinationSum4(nums, target)
    return 1000 * (time.perf_counter() - t) / lan

random.seed(1377)
nums = random.sample(range(1, 1001), 200)
sys.setrecursionlimit(20000)
for ten, cls in [("memo", SolutionMemo), ("DP", SolutionDP), ("DP sắp xếp", SolutionDPSorted)]:
    print(f"{ten:<12}: {do_thoi_gian(cls, nums, 1000):.2f} ms / lần")
sys.setrecursionlimit(1000)

memo        : 19.47 ms / lần
DP          : 8.61 ms / lần


DP sắp xếp  : 6.23 ms / lần


## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `combinationSum4([2], 6)`; (b) `combinationSum4([1, 2], 3)`; (c) `combinationSum4([5], 3)`; (d) `dp` cuối cùng khi `nums = [1, 2]`, `target = 4` (cả mảng); (e) `math.factorial(4) // (math.factorial(2) * math.factorial(2))`.

**Bài 2 (tính tay).** Với `nums = [1, 3, 4]`, `target = 6`: lập bảng `dp[0..6]` như mục 8 và liệt kê **tất cả** chuỗi tổng `6`.

**Bài 3 (điền TODO).** Hoàn thành `count_todo(nums, target)` theo Cách 3.

In [31]:
def count_todo(nums, target):
    # TODO 1: tạo dp = [0] * (target + 1) và gán dp[0] = 1
    # TODO 2: vòng ngoài: for s in range(1, target + 1)
    # TODO 3: vòng trong: for x in nums, nếu x <= s thì dp[s] += dp[s - x]
    # TODO 4: return dp[target]
    return None

# KHÔNG gọi count_todo ở đây để "Run All" không lỗi (chưa điền xong).

**Bài 4 (tự viết lại).** Viết `count_memo(nums, target)` bằng `lru_cache` **mà không nhìn lại** mục 7. Kiểm tra `([1,2,3], 4) → 7`.

**Bài 5 (biến thể / mini challenge).** (a) **Coin Change II**: viết `coin_combinations(coins, amount)` đếm **tổ hợp** (không thứ tự); kiểm tra `([1,2,5], 5) → 4`. (b) Viết `count_exact_k(nums, target, k)` đếm chuỗi có **đúng `k` phần tử**; kiểm tra `([1,2,3], 4, 2) → 3` (các chuỗi `(1,3)`, `(3,1)`, `(2,2)`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: (a) `1` (chỉ `(2,2,2)`), (b) `3` (`(1,1,1)`, `(1,2)`, `(2,1)`), (c) `0`, (d) `[1, 1, 2, 3, 5]`, (e) `6`. Bài 3: nhớ `dp[0] = 1` và `if x <= s`.
- **Mức 2 (vừa):** Bài 2: `dp = [1, 1, 1, 2, 4, 6, 9]`; cách nhớ: `dp[s] = dp[s-1] + dp[s-3] + dp[s-4]`. Bài 4: `dfs(rem)`; `rem == 0 → 1`, `rem < 0 → 0`.
- **Mức 3 (sâu):** Bài 5(a): **phần tử ngoài, tổng trong** (`for c in coins: for s in range(c, amount + 1): dp[s] += dp[s - c]`). Bài 5(b): `dp[j][s]` = số chuỗi dài đúng `j` có tổng `s`; `dp[j][s] = Σ_x dp[j-1][s-x]`.

## 19. Đáp án tham khảo cho bài tập

In [32]:
# Bài 1
assert Solution().combinationSum4([2], 6) == 1
assert Solution().combinationSum4([1, 2], 3) == 3
assert Solution().combinationSum4([5], 3) == 0
dp12 = [1, 0, 0, 0, 0]
for s in range(1, 5):
    for x in (1, 2):
        if x <= s:
            dp12[s] += dp12[s - x]
assert dp12 == [1, 1, 2, 3, 5]
assert math.factorial(4) // (math.factorial(2) * math.factorial(2)) == 6
print("Bài 1 OK")

# Bài 2
dp = [1] + [0] * 6
for s in range(1, 7):
    for x in (1, 3, 4):
        if x <= s:
            dp[s] += dp[s - x]
print("dp =", dp)
assert dp == [1, 1, 1, 2, 4, 6, 9]
cs = sinh_chuoi([1, 3, 4], 6)
print(len(cs), "chuỗi:", sorted(cs))
assert len(cs) == 9
print("Bài 2 OK")

# Bài 3
def count_todo_ans(nums, target):
    dp = [0] * (target + 1)
    dp[0] = 1
    for s in range(1, target + 1):
        for x in nums:
            if x <= s:
                dp[s] += dp[s - x]
    return dp[target]

random.seed(5)
for _ in range(200):
    nums = random.sample(range(1, 9), random.randint(1, 4))
    target = random.randint(1, 15)
    assert count_todo_ans(nums, target) == comb_brute(nums, target)
print("Bài 3 OK")

# Bài 4
def count_memo(nums, target):
    @functools.lru_cache(maxsize=None)
    def dfs(rem):
        if rem == 0:
            return 1
        if rem < 0:
            return 0
        return sum(dfs(rem - x) for x in nums)
    return dfs(target)

assert count_memo([1, 2, 3], 4) == 7 and count_memo([9], 3) == 0
print("Bài 4 OK")

# Bài 5
def coin_combinations(coins, amount):
    dp = [0] * (amount + 1)
    dp[0] = 1
    for c in coins:                           # đồng xu ở vòng ngoài => tổ hợp
        for s in range(c, amount + 1):
            dp[s] += dp[s - c]
    return dp[amount]

assert coin_combinations([1, 2, 5], 5) == 4

def count_exact_k(nums, target, k):
    dp = [[0] * (target + 1) for _ in range(k + 1)]
    dp[0][0] = 1
    for j in range(1, k + 1):
        for s in range(target + 1):
            for x in nums:
                if x <= s:
                    dp[j][s] += dp[j - 1][s - x]
    return dp[k][target]

assert count_exact_k([1, 2, 3], 4, 2) == 3
for target in range(1, 12):                                  # tổng theo mọi k phải bằng đáp án gốc
    assert sum(count_exact_k([1, 2, 3], target, k) for k in range(1, target + 1)) == Solution().combinationSum4([1, 2, 3], target)
print("Bài 5 OK")

Bài 1 OK
dp = [1, 1, 1, 2, 4, 6, 9]
9 chuỗi: [(1, 1, 1, 1, 1, 1), (1, 1, 1, 3), (1, 1, 3, 1), (1, 1, 4), (1, 3, 1, 1), (1, 4, 1), (3, 1, 1, 1), (3, 3), (4, 1, 1)]
Bài 2 OK
Bài 3 OK
Bài 4 OK
Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (đếm hoán vị — tổng ngoài):**

```python
dp = [0] * (target + 1)
dp[0] = 1
for s in range(1, target + 1):
    for x in nums:
        if x <= s:
            dp[s] += dp[s - x]
return dp[target]
```

**Template (đếm tổ hợp — đồng xu ngoài, Coin Change II):**

```python
for c in coins:
    for s in range(c, amount + 1):
        dp[s] += dp[s - c]
```

**Checklist:**

- [ ] Hỏi: thứ tự có quan trọng không? Có → **tổng ngoài**; không → **phần tử ngoài**.
- [ ] `dp[0] = 1`.
- [ ] Có điều kiện `x <= s` (hoặc sắp xếp + `break`) — nhớ Python **không báo lỗi** khi chỉ số âm.
- [ ] Không dùng đệ quy sâu `1000` tầng nếu không tăng giới hạn; ưu tiên bottom-up.
- [ ] Thử `([1,2,3], 4) → 7`, `([9], 3) → 0`, `([1], 1000) → 1`.
- [ ] Đúng signature `class Solution` / `combinationSum4(self, nums: List[int], target: int) -> int`.
- [ ] Follow up số âm: nhắc cần giới hạn độ dài chuỗi.